# 🌊 Notebook 1: Why You Need a High-Water Mark

**The setup:** A leader replicates a log to several followers. Clients want to read what the leader has accepted. The naive choice — *let clients read every entry the leader writes locally* — leads to **lost writes** when the leader crashes before its followers caught up.

We'll simulate it.


## 🛠️ Setup

```bash
cd 02-distributed-primitives/high-water-mark
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## 🟥 BAD: leader exposes its tail before followers ack

In [ ]:
from dataclasses import dataclass, field
from typing import List

@dataclass
class Node:
    name: str
    log: List[str] = field(default_factory=list)

leader = Node('leader')
followers = [Node('f1'), Node('f2')]

def replicate(entry, deliver_to):
    leader.log.append(entry)
    # Only some followers receive the entry before the leader crashes
    for f in deliver_to:
        f.log.append(entry)

# Three writes; the third never reaches any follower
replicate('A', followers)        # everyone has A
replicate('B', followers[:1])    # only f1 has B
replicate('C', [])               # no one but leader has C

# Client reads C from the leader and acts on it (e.g. confirms an order)
client_saw = leader.log[-1]
print('client read:', client_saw)

# Leader crashes. We must elect a new leader from followers.
new_leader = max(followers, key=lambda f: len(f.log))
print('new leader log:', new_leader.log)
print('lost write?', client_saw not in new_leader.log)


The client *believes* `C` is committed but the new leader has no record of it. **Data loss.**

👉 Next notebook: track a **high-water mark** — the highest offset replicated to enough followers — and only let clients read up to that mark.